# Modelado de Primer Orden: Vehículo RC de Alta Eficiencia
**Asignatura:** Proyectos CDIO  
**Departamento:** Ingeniería Mecánica, Facultad de Ingeniería, Universidad de Concepción.

---

## 1. Contexto y Misión General (Nivel 0)
Este documento presenta el desarrollo de un modelo matemático de primer orden para el diseño y optimización de un vehículo a control remoto (RC). El foco principal de la metodología de diseño radica en maximizar la relación potencia/masa, garantizando la integridad estructural y la estabilidad dinámica bajo condiciones críticas de operación.

### Misión del Sistema
Transportar la masa propia del vehículo RC (chasis impreso, componentes electrónicos, motor DC RS550 y batería LiPo 4S) más una carga útil determinada ($m_{\text{payload}}$ = TBD kg) de manera terrestre por una pista recta, optimizando la eficiencia estructural y alcanzando la velocidad máxima operacional de diseño de $7.6 \text{ m/s}$.

---

## 2. Descomposición Jerárquica del Sistema (Heurística $7 \pm 2$)
Para gestionar la complejidad inherente al aumento de fidelidad del modelo físico, el vehículo se descompone jerárquicamente en **5 subsistemas principales**, cumpliendo con la heurística de diseño que limita la carga cognitiva entre $5$ y $9$ elementos por nivel:

1. **Subsistema Estructural y Chasis:** Estructura soportante e interfaces físicas. A partir de simulaciones avanzadas por el método de elementos finitos (FEM), se determinó que el material óptimo para el chasis es el polímero **PLA**, ya que soporta de manera correcta el torque torsional y las fuerzas de inercia de la carrera.
2. **Subsistema de Propulsión (Planta Motriz):** Motor de corriente continua (DC) RS550 y tren de engranajes de reducción (piñón y corona).
3. **Subsistema de Gestión de Energía:** Batería de polímero de litio (LiPo) de 4 celdas (4S) y circuitos de regulación de voltaje (RV).
4. **Subsistema de Control y Electrónica:** Microprocesador Arduino, receptor de potencia/puente H (RP), módulo de comunicación Bluetooth (Bt) y la placa de circuito impreso (PCB) personalizada.
5. **Subsistema de Dirección y Actuación:** Servoactuador de dirección y varillajes mecánicos para el control cinemático del eje delantero.

---

## 3. Modelo Input-Process-Output (IPO) y Relaciones Físicas

### Diagrama de Flujo Lógico
A continuación, se presenta la cadena de bloques lógicos que gobierna las ecuaciones de primer orden del vehículo, cuyo archivo editable original se encuentra adjunto en el repositorio como `diagrama_logica.vsdx`:

![Diagrama de Flujo Conceptual](./images/diagrama_flujo.png)

### Ecuaciones del Modelo (Formato LaTeX)
Para evaluar el desempeño dinámico en condiciones estáticas de operación y de partida, se plantean las ecuaciones fundamentales de la mecánica y de máquinas eléctricas.

La **Fuerza de Tracción** ($F_{\text{trac}}$) disponible en las ruedas motrices generada por el torque del motor está dada por la relación:
$$F_{\text{trac}} = \frac{\tau \cdot N}{r}$$

Donde:
* $\tau$ es el torque del motor DC RS550 en el eje ($\text{Nm}$).
* $N$ es la relación de transmisión del reductor mecánica (adimensional).
* $r$ es el radio efectivo de la rueda motriz ($\text{m}$).

Asumiendo condiciones de partida ideales en plano sobre un Diagrama de Cuerpo Libre (DCL), la **Aceleración Inicial** ($a$) se calcula aplicando la Segunda Ley de Newton:
$$a = \frac{F_{\text{trac}}}{m_{\text{total}}}$$

Donde $m_{\text{total}}$ es la masa sumada del chasis, la electrónica, los actuadores y la carga útil. Finalmente, la **Potencia Mecánica** ($P_{\text{mec}}$) desarrollada por la planta motriz en función de la velocidad lineal ($v$) se modela como:
$$P_{\text{mec}} = F_{\text{trac}} \cdot v$$

In [1]:
# =====================================================================
# CÓDIGO PYTHON EJECUTABLE: MODELO DE PRIMER ORDEN - VEHÍCULO RC
# =====================================================================

def evaluar_dinamica_rc(torque_nm, masa_kg, radio_rueda_m, relacion_transmision, velocidad_diseno):
    """
    Función que implementa el bloque 'Process' del modelo IPO.
    Calcula la fuerza de tracción, aceleración teórica y potencia mecánica.
    """
    # 1. Proceso matemático de las ecuaciones de primer orden
    fuerza_traccion = (torque_nm * relacion_transmision) / radio_rueda_m
    aceleracion_inicial = fuerza_traccion / masa_kg
    potencia_mecanica = fuerza_traccion * velocidad_diseno
    
    # 2. Bloque de Salidas (Outputs)
    return {
        "Fuerza de Tracción Disponible [N]": round(fuerza_traccion, 2),
        "Aceleración Inicial Estimada [m/s²]": round(aceleracion_inicial, 2),
        "Potencia a Velocidad de Diseño [W]": round(potencia_mecanica, 2)
    }

# --- Bloque de Entradas (Inputs) ---
# Se utilizan los parámetros de diseño del vehículo y curvas del motor RS550
inputs_vehiculo = {
    "torque_nm": 0.30,               # 300 mNm máximo operacional según ficha técnica del motor
    "masa_kg": 1.15,                 # Masa estimada (Chasis PLA + electrónica + batería 4S)
    "radio_rueda_m": 0.032,          # Radio de rueda estándar de 32 mm
    "relacion_transmision": 3.5,     # Relación de reducción piñón/corona sugerida
    "velocidad_diseno": 7.6          # Velocidad máxima operacional objetivo (m/s)
}

# Ejecución del modelo numérico
outputs_vehiculo = evaluar_dinamica_rc(**inputs_vehiculo)

# Despliegue formal de resultados en consola
print("=======================================================")
print("          RESULTADOS DE LA SIMULACIÓN NUMÉRICA         ")
print("=======================================================")
for variable, valor in outputs_vehiculo.items():
    print(f"-> {variable}: {valor}")
print("=======================================================")

          RESULTADOS DE LA SIMULACIÓN NUMÉRICA         
-> Fuerza de Tracción Disponible [N]: 32.81
-> Aceleración Inicial Estimada [m/s²]: 28.53
-> Potencia a Velocidad de Diseño [W]: 249.38
